In [1]:
"""Stage B — export an EMPTY template (titles/format only) to review the layout."""
import importlib
import fit_excel_export as fxe
importlib.reload(fxe)  # pick up edits without restarting the kernel

# Write the empty template; uses default SystematicConfig values.
path = fxe.export_empty_template(out_dir="../output/excel_export")
print("Wrote:", path.resolve())


Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_empty_template.xlsx


In [2]:
"""Stage C — Cell 1: setup, load ONE test discharge via pickle cache."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, curve_fit

# --- locate project root so pha_lib is importable ---
def _find_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "pha_lib").exists():
            return p
    raise FileNotFoundError("pha_lib not found")

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pha_lib import io
from pha_lib.timetrace import integrate_energy_window
from pha_lib.discharges import detect_injections

# --- parameters (same as sigma clipping.ipynb) ---
DISCHARGE_ID = "2025-04-29--10-19"
DISCHARGE_DIR = ROOT / "data" / "discharges" / DISCHARGE_ID
CHANNEL_ID = 2
LINE_E = 6660.0   # eV
HALF_W = 60.0     # eV half-window

# --- load discharge through a pickle cache (build it once if missing) ---
cache_file = DISCHARGE_DIR / "cache" / "discharge_object.pkl"
if cache_file.exists():
    discharge = pd.read_pickle(cache_file)            # fast path: reuse cache
    print(f"Loaded from pickle: {cache_file}")
else:
    discharge = io.load_test_folder(DISCHARGE_DIR, discharge_id=DISCHARGE_ID)
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(discharge, cache_file)               # cache for next time
    print(f"Built pickle cache: {cache_file}")

# --- detect injections on the chosen channel/window ---
channel = discharge.channels[CHANNEL_ID]
trace = integrate_energy_window(channel, LINE_E, HALF_W)
injections = detect_injections(trace, LINE_E)
print(f"{discharge.discharge_id}: {len(injections)} injection(s) detected")


Loaded from pickle: D:\ℱ Sci\IFPiLM\1st task\data\discharges\2025-04-29--10-19\cache\discharge_object.pkl
2025-04-29--10-19: 4 injection(s) detected


In [3]:
"""Stage C — Cell 2: sigma-clip fit code (copied from sigma clipping.ipynb)."""

_MAD_SCALE = 1.4826   # makes MAD equivalent to 1-sigma for Gaussian noise


def _exp_model(t, A, tau, t_0, C_bg):
    """Exponential decay: A * exp(-(t - t_0) / tau) + C_bg."""
    return A * np.exp(-(t - t_0) / tau) + C_bg


def _initial_guess(y_fit, t_0, C_bg):
    """Simple initial guess: A from first point above background, tau=1."""
    A0 = max(float(y_fit[0] - C_bg), 1.0)
    return [A0, 1.0]


def find_first_falling_frame(trace, injection) -> int:
    """Return the first frame of the LAST descent inside the injection span."""
    fn = trace.frame_numbers
    v = trace.values

    try:
        s_idx = int(np.where(fn == int(injection.start_frame))[0][0])
    except IndexError:
        return int(injection.start_frame + 1)

    try:
        e_idx = int(np.where(fn == int(injection.finish_frame))[0][0])
    except IndexError:
        e_idx = len(fn) - 1

    # scan forward, keep the LAST local maximum (rightmost roller-coaster top)
    last_peak_idx = None
    for i in range(s_idx, e_idx):
        not_lower_than_prev = (i == s_idx) or (v[i] >= v[i - 1])
        higher_than_next = v[i] > v[i + 1]
        if not_lower_than_prev and higher_than_next:
            last_peak_idx = i

    if last_peak_idx is not None:
        next_idx = last_peak_idx + 1
        if next_idx <= e_idx:
            return int(fn[next_idx])

    # fallback 1: first falling step anywhere in span
    for i in range(s_idx + 1, e_idx + 1):
        if v[i] < v[i - 1]:
            return int(fn[i])

    # fallback 2: monotonically non-decreasing
    return int(injection.start_frame + 1)


def _fail(msg, t_0, C_bg):
    """Return a standardized failure dict."""
    nan = float("nan")
    return {
        "success": False, "message": msg,
        "A": nan, "tau": nan, "t_0": t_0, "C_bg": C_bg,
        "A_err": nan, "tau_err": nan,
        "inlier_mask": None, "t_all": None, "y_all": None,
        "n_total": 0, "n_inliers": 0, "inlier_fraction": nan,
        "rss_inliers": nan, "rmse_inliers": nan,
        "reduced_chi2": nan, "tau_err_rel": nan, "iterations": 0,
    }


def fit_sigma_clip(trace, injection, sigma_thresh: float = 1,
                   max_iter: int = 10, min_inliers: int = 4) -> dict:
    """Robust iterative sigma-clipping fit in log space over the decay span."""
    fn = trace.frame_numbers
    v = trace.values

    # t_0 = first frame of last descent; C_bg = median background
    t_0 = float(find_first_falling_frame(trace, injection))
    C_bg = float(np.median(v))

    # locate decay span [t_0 .. finish_frame]
    try:
        start_idx = int(np.where(fn == int(t_0))[0][0])
    except IndexError:
        return _fail(f"t_0={int(t_0)} not in trace", t_0, C_bg)
    try:
        end_idx = int(np.where(fn == int(injection.finish_frame))[0][0]) + 1
    except IndexError:
        end_idx = len(fn)
    end_idx = max(end_idx, start_idx + min_inliers)

    t_all = fn[start_idx:end_idx].astype(float)
    y_all = v[start_idx:end_idx].astype(float)
    n_total = len(t_all)
    if n_total < min_inliers:
        return _fail(f"decay span too short ({n_total} pts)", t_0, C_bg)

    # only points above background are valid in log space
    valid_mask = y_all > C_bg
    if valid_mask.sum() < min_inliers:
        return _fail(f"too few points above C_bg ({valid_mask.sum()})", t_0, C_bg)

    z_all = np.where(valid_mask, np.log(np.maximum(y_all - C_bg, 1e-300)), np.nan)

    # initial log-space linear seed (slope must be negative for decay)
    t_valid = t_all[valid_mask]
    z_valid = z_all[valid_mask]
    try:
        m_seed, b_seed = np.polyfit(t_valid, z_valid, 1)
        if m_seed >= 0:
            m_seed = -0.1
    except Exception:
        m_seed = -0.1
        b_seed = float(np.log(max(y_all[0] - C_bg, 1.0)))

    m_fit, b_fit = m_seed, b_seed

    # iterative sigma-clipping in log space (all points equal leverage)
    inlier_mask = valid_mask.copy()
    for iteration in range(max_iter):
        t_in = t_all[inlier_mask]
        z_in = z_all[inlier_mask]
        if len(t_in) < min_inliers:
            break
        try:
            m_fit, b_fit = np.polyfit(t_in, z_in, 1)
            if m_fit >= 0:
                m_fit = m_seed
        except Exception:
            break

        z_pred_all = m_fit * t_all + b_fit
        resid_all = z_all - z_pred_all
        mad = _MAD_SCALE * np.median(np.abs(resid_all[inlier_mask]))
        if mad < 1e-12:
            break
        new_mask = (np.abs(resid_all) <= sigma_thresh * mad) & valid_mask
        if new_mask.sum() < min_inliers:
            new_mask = inlier_mask
        if np.array_equal(new_mask, inlier_mask):
            inlier_mask = new_mask
            break
        inlier_mask = new_mask
        m_seed = m_fit

    # final OLS fit in original space for proper errors
    t_in = t_all[inlier_mask]
    y_in = y_all[inlier_mask]
    n_in = int(inlier_mask.sum())
    if n_in < min_inliers:
        return _fail(f"too few inliers ({n_in}) after clipping", t_0, C_bg)

    tau_seed = max(-1.0 / m_fit, 0.01) if m_fit < 0 else 1.0
    A_seed = max(float(np.exp(b_fit + m_fit * t_0)), 1e-6)
    try:
        popt, pcov = curve_fit(
            lambda t, A, tau: _exp_model(t, A, tau, t_0, C_bg),
            t_in, y_in, p0=[A_seed, tau_seed], maxfev=5000,
        )
    except Exception as exc:
        return _fail(f"final fit failed: {exc}", t_0, C_bg)

    A_fit, tau_fit = float(popt[0]), float(popt[1])
    diag = np.diag(pcov).astype(float)
    diag[diag < 0] = np.nan
    A_err, tau_err = float(np.sqrt(diag[0])), float(np.sqrt(diag[1]))

    resid_in = y_in - _exp_model(t_in, A_fit, tau_fit, t_0, C_bg)
    rss = float(np.sum(resid_in ** 2))
    dof = n_in - 2
    rmse = float(np.sqrt(rss / n_in))
    red_chi2 = float(rss / dof) if dof > 0 else np.nan

    return {
        "success": True, "A": A_fit, "tau": tau_fit, "t_0": t_0, "C_bg": C_bg,
        "A_err": A_err, "tau_err": tau_err, "inlier_mask": inlier_mask,
        "t_all": t_all, "y_all": y_all, "n_total": n_total, "n_inliers": n_in,
        "inlier_fraction": n_in / n_total, "rss_inliers": rss, "rmse_inliers": rmse,
        "reduced_chi2": red_chi2,
        "tau_err_rel": tau_err / abs(tau_fit) if tau_fit != 0 else np.nan,
        "message": f"ok (t_0={int(t_0)}, iter={iteration+1}, inliers={n_in}/{n_total})",
        "iterations": iteration + 1,
    }

print("fit_sigma_clip and helpers defined.")


fit_sigma_clip and helpers defined.


In [4]:
"""Stage C — Cell 3: fit every injection, build a plot, export to Excel."""
import importlib
import fit_excel_export as fxe
importlib.reload(fxe)


def make_fit_ax(trace, injection, result):
    """Build a small data+fit figure for one injection; return (fig, ax)."""
    fig, ax = plt.subplots(figsize=(3.2, 2.1))
    # plot the full injection span as faint points for context
    fn, v = trace.frame_numbers, trace.values
    span = (fn >= injection.start_frame) & (fn <= injection.finish_frame)
    ax.plot(fn[span], v[span], "o", ms=3, color="0.7", label="all")
    if result["success"]:
        # highlight inliers and overlay the fitted exponential curve
        t_all, y_all = result["t_all"], result["y_all"]
        mask = result["inlier_mask"]
        ax.plot(t_all[mask], y_all[mask], "o", ms=3, color="tab:blue", label="inliers")
        tt = np.linspace(t_all.min(), t_all.max(), 100)
        yy = fxe_exp(tt, result)   # fitted curve helper below
        ax.plot(tt, yy, "-", color="tab:red", lw=1.2, label="fit")
    ax.set_xlabel("frame")
    ax.set_ylabel("events")
    ax.set_title(f"inj #{injection.injection_no}", fontsize=8)
    fig.tight_layout()
    return fig, ax


def fxe_exp(t, result):
    """Evaluate the fitted exponential from a result dict."""
    return _exp_model(t, result["A"], result["tau"], result["t_0"], result["C_bg"])


# --- fit each injection and collect export rows ---
rows = []
for inj in injections:
    res = fit_sigma_clip(trace, inj, sigma_thresh=1, min_inliers=4, max_iter=10)
    fig, ax = make_fit_ax(trace, inj, res)
    rows.append({"discharge_id": discharge.discharge_id,
                 "injection": inj, "result": res, "ax": ax})
    plt.close(fig)   # keep notebook output clean; ax still renders to PNG

# --- export to Excel ---
path = fxe.export_fits_to_excel(
    rows, out_dir="../output/excel_export", filename="fits_stageC.xlsx",
)
print("Wrote:", path.resolve(), f"({len(rows)} injection rows)")


Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_stageC.xlsx (4 injection rows)


In [5]:
"""Stage D — load ALL discharges via pickle, sigma-fit, export to one sheet."""
importlib.reload(fxe)

DISCHARGES_ROOT = ROOT / "data" / "discharges"


def load_discharge_cached(folder: Path):
    """Load a discharge from its pickle cache, building the cache if missing."""
    cache = folder / "cache" / "discharge_object.pkl"
    if cache.exists():
        return pd.read_pickle(cache)                  # fast path
    disc = io.load_test_folder(folder, discharge_id=folder.name)
    cache.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(disc, cache)                          # build cache once
    return disc


def fit_discharge_rows(disc):
    """Detect injections on the chosen channel/window, fit each, return rows."""
    ch = disc.channels[CHANNEL_ID]
    tr = integrate_energy_window(ch, LINE_E, HALF_W)
    rows_local = []
    for inj in detect_injections(tr, LINE_E):
        res = fit_sigma_clip(tr, inj, sigma_thresh=1, min_inliers=4, max_iter=10)
        fig, ax = make_fit_ax(tr, inj, res)
        rows_local.append({"discharge_id": disc.discharge_id,
                           "injection": inj, "result": res, "ax": ax})
        plt.close(fig)
    return rows_local


# --- collect rows across every discharge folder (skip the 'test' folder) ---
all_rows = []
folders = sorted(p for p in DISCHARGES_ROOT.iterdir()
                 if p.is_dir() and p.name != "test")
for folder in folders:
    disc = load_discharge_cached(folder)
    disc_rows = fit_discharge_rows(disc)
    all_rows.extend(disc_rows)
    print(f"{folder.name}: {len(disc_rows)} injection(s)")

# --- export all discharges into one Excel sheet ---
path = fxe.export_fits_to_excel(
    all_rows, out_dir="../output/excel_export", filename="fits_stageD.xlsx",
)
print(f"\nWrote: {path.resolve()}  ({len(all_rows)} rows from {len(folders)} discharges)")


2025-04-29--10-19: 4 injection(s)
2025-04-29--10-24: 4 injection(s)
2025-04-29--10-42: 1 injection(s)
2025-04-29--10-49: 1 injection(s)
2025-04-29--11-17: 3 injection(s)


C:\Users\Superadmin\AppData\Local\Temp\ipykernel_18528\1045894273.py:153: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, pcov = curve_fit(


2025-04-29--11-19: 5 injection(s)
2025-04-29--11-23: 3 injection(s)
2025-04-29--11-32: 3 injection(s)
2025-04-29--11-37: 1 injection(s)
2025-04-29--12-14: 3 injection(s)
2025-04-29--12-28: 5 injection(s)
2025-04-29--12-50: 2 injection(s)
2025-04-29--12-58: 4 injection(s)
2025-04-29--13-17: 2 injection(s)
2025-04-29--13-51: 5 injection(s)

Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_stageD.xlsx  (46 rows from 15 discharges)


In [6]:
"""Stage E — export only 3 discharges, reusing rows already in memory."""
importlib.reload(fxe)

# Pick the first 3 discharge ids (preserve their original order).
first_three_ids = []
for r in all_rows:
    if r["discharge_id"] not in first_three_ids:
        first_three_ids.append(r["discharge_id"])
    if len(first_three_ids) == 3:
        break

# Reuse the already-computed rows for those discharges (no refitting).
three_rows = [r for r in all_rows if r["discharge_id"] in first_three_ids]

path = fxe.export_fits_to_excel(
    three_rows, out_dir="../output/excel_export", filename="fits_stageE.xlsx",
)
print(f"Discharges: {first_three_ids}")
print(f"Wrote: {path.resolve()}  ({len(three_rows)} rows)")


Discharges: ['2025-04-29--10-19', '2025-04-29--10-24', '2025-04-29--10-42']
Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_stageE.xlsx  (9 rows)


In [7]:
"""Stage F — export the same 3 discharges WITH colouring (reuse memory)."""
importlib.reload(fxe)

# three_rows already holds the first 3 discharges' fits from Stage E.
path = fxe.export_fits_to_excel(
    three_rows, out_dir="../output/excel_export",
    filename="fits_stageF_colored.xlsx", colorize=True,
)
print(f"Wrote: {path.resolve()}  ({len(three_rows)} rows, coloured)")


Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_stageF_colored.xlsx  (9 rows, coloured)


In [8]:
"""Stage G — export ALL discharges WITH colouring (reuse memory)."""
importlib.reload(fxe)

# all_rows already holds every discharge's fits from Stage D.
path = fxe.export_fits_to_excel(
    all_rows, out_dir="../output/excel_export",
    filename="fits_stageG_colored.xlsx", colorize=True,
)
print(f"Wrote: {path.resolve()}  ({len(all_rows)} rows, coloured)")


Wrote: D:\ℱ Sci\IFPiLM\1st task\output\excel_export\fits_stageG_colored.xlsx  (46 rows, coloured)
